# Exploring the BPE tokenizer

Walk through the byte-level BPE pipeline built in `cs336.tokenizer`:
pre-tokenization → training (learning merges) → encode/decode.

Kernel: **CS336 (.venv)**. If imports fail, pick that kernel (top-right).

In [ ]:
from cs336.tokenizer import train_bpe, BPETokenizer
from cs336.tokenizer.bpe import pretokenize, count_pairs, merge_pair
print('imports OK')

## 1. Pre-tokenization

The GPT-2 regex splits text into word-ish chunks *before* BPE runs, so merges never
cross word/punctuation boundaries. Note the leading spaces attached to words.

In [ ]:
pretokenize("Don't merge across words! 123 abc")

## 2. Train BPE and inspect the learned merges

Each merge combines the most frequent adjacent pair (ties → lexicographically greatest).

In [ ]:
corpus = (
    "the cat sat on the mat. "
    "the cat sat. the mat sat. "
    "a cat, a mat, a hat."
) * 20

vocab, merges = train_bpe(corpus, vocab_size=300, special_tokens=["<|endoftext|>"])
print(f"vocab size: {len(vocab)}   num merges: {len(merges)}")
print("\nfirst 15 merges (in learning order):")
for i, (a, b) in enumerate(merges[:15]):
    print(f"  {i:2d}: {a!r} + {b!r} -> {a + b!r}")

## 3. Encode / decode roundtrip

In [ ]:
tok = BPETokenizer(vocab, merges, special_tokens=["<|endoftext|>"])

text = "the cat sat on the mat"
ids = tok.encode(text)
print("text  :", text)
print("ids   :", ids)
print("tokens:", [vocab[i] for i in ids])
print("decode:", repr(tok.decode(ids)))
print("roundtrip ok:", tok.decode(ids) == text)
print(f"compression: {len(text.encode())} bytes -> {len(ids)} tokens")

## 4. Special tokens stay atomic

`<|endoftext|>` is emitted as a single id and is never split or merged into.

In [ ]:
ids = tok.encode("the cat<|endoftext|>the mat")
eot = next(i for i, b in vocab.items() if b == b"<|endoftext|>")
print("eot id:", eot)
print("ids   :", ids)
print("tokens:", [vocab[i] for i in ids])
print("decode:", repr(tok.decode(ids)))

## 5. Byte-level = handles any Unicode

Even text the tokenizer never trained on roundtrips, because the base vocab is the 256 bytes.

In [ ]:
s = "héllo 世界 🚀"
print(repr(tok.decode(tok.encode(s))), "==", repr(s), "->", tok.decode(tok.encode(s)) == s)